In [ ]:
import csv
import numpy as np
import pandas as pd
import sys, os
import re
import ast
import datetime as dt
from tqdm import tqdm
from collections import defaultdict, Counter

import warnings

warnings.filterwarnings("ignore")

In [ ]:
way = 'max'  

def median_chart_process(ts):
    ts.loc[:, (ts == 0).all(axis=0)] = np.nan  

    processed_values = {}
    for col in ts.columns:
        if col.startswith('CHART_Cate_'):  

            processed_values[col] = ts[col].max()
        elif col.startswith('CHART_Str_'):  

            non_zero_values = ts[col][ts[col] != '0']
            if not non_zero_values.empty:
                processed_values[col] = non_zero_values.mode().iloc[0]
            else:
                processed_values[col] = 0
        else:

            if way == 'median':
                processed_values[col] = ts[col].median()
            elif way == 'min':
                processed_values[col] = ts[col].min()
            else: 
                processed_values[col] = ts[col].max()
    

    result = pd.DataFrame([processed_values])
    
    return result

In [ ]:
all_ts = pd.DataFrame()

for i in tqdm(os.listdir('/MIMIC_IV/Final_Chart_2/')):

    file_path = f'/MIMIC_IV/Final_Chart_2/{i}'
    ts = pd.read_csv(file_path)
    
    if ts.shape[0]>0:
        ts = ts.iloc[:96,:]

        ts = median_chart_process(ts)

        ts['ICUSTAY_ID'] = i.split('.')[0]

        all_ts = pd.concat([all_ts, ts], ignore_index=True)
    
all_ts = all_ts.dropna(axis=1, how='all')

100%|████████████████████████████████████████████████████████████████████████████| 74827/74827 [54:50<00:00, 22.74it/s]


In [ ]:
all_ts.to_csv('/MIMIC_IV/Median_B/A24_Chart_median_max_nan.csv',index=False)